# LLM API calls (250) to enhance the quality of silver labels

고려대학교 보건과학대학 바이오의공학부

2021250031 정예준

In [ ]:
import os
from google import genai
from google.genai import types
from google.genai.types import GenerateContentConfig

In [ ]:
GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]

In [ ]:
import torch

print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device name:", torch.cuda.get_device_name(0))

cuda available: False


In [ ]:
# Import libraries
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset
import copy
import random
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Random seed
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

## 1. Load data

In [ ]:
# Root dataset directory
ROOT = Path("Amazon_products")

# Corpus paths
TRAIN_CORPUS_PATH = ROOT / "train" / "train_corpus.txt"
TEST_CORPUS_PATH = ROOT / "test" / "test_corpus.txt"

# Class-related information
CLASS_HIERARCHY_PATH = ROOT / "class_hierarchy.txt"
CLASS_KEYWORDS_PATH = ROOT / "class_related_keywords.txt"
CLASS_NAMES_PATH = ROOT / "classes.txt"

# Pre-trained embeddings (fine-tuned)
# (In "generate_embeddings_fine-tuning.ipynb" / No need to reproduce that file)
LABEL_EMB_PATH = ROOT / "label_bert_mean_dapt.pt"
TRAIN_EMB_PATH = ROOT / "train_bert_mean_dapt.pt"
TEST_EMB_PATH  = ROOT / "test_bert_mean_dapt.pt"

In [ ]:
# Data loading function
def load_corpus(path):
    """
    Load corpus file (train/test).
    Each line: `<int_id> <space> <review text...>`
    Returns: {doc_id: text}
    """
    corpus = {}
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            doc_id_str, text = line.split(maxsplit=1)
            doc_id = int(doc_id_str)
            corpus[doc_id] = text.strip()
    return corpus

def load_class_names(path):
    """
    Load class name and id.
    Each line: `<id> <class_name>`
    Returns:
      - class_names: index == class_id (list)
      - name_to_id : class_name -> class_id (dictionary)
    """
    class_names = []
    name_to_id = {}

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            cls_id = int(parts[0])
            cls_name = parts[1]

            class_names.append(cls_name)
            name_to_id[cls_name] = cls_id

    return class_names, name_to_id

def load_class_hierarchy(path):
    """
    Load class hierarchy edges.
    Each line: `<parent_id> <child_id>`
    Returns:
      - parent_to_children: {parent_id: [child_id, ...]}
      - child_to_parents: {child_id: [parent_id, ...]}
    """
    parent_to_children = defaultdict(list)
    child_to_parents = defaultdict(list)

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parent_str, child_str = line.split()
            parent = int(parent_str)
            child = int(child_str)
            parent_to_children[parent].append(child)
            child_to_parents[child].append(parent)

    return dict(parent_to_children), dict(child_to_parents)

def load_class_keywords(path, class_names):
    """
    Load class-related keywords.
    Each line: `<class_name>:kw1,kw2,...`
    Returns: {class_id: [kw1, kw2, ...]}
    """
    name_to_id = {name: idx for idx, name in enumerate(class_names)}
    class_keywords = {}

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            cls_name, kws_str = line.split(":", maxsplit=1)
            cls_name = cls_name.strip()
            kws = [k.strip() for k in kws_str.split(",") if k.strip()]
            cls_id = name_to_id[cls_name]
            class_keywords[cls_id] = kws

    return class_keywords

In [ ]:
# Load data
train_corpus = load_corpus(TRAIN_CORPUS_PATH)
test_corpus  = load_corpus(TEST_CORPUS_PATH)
class_names, name_to_id = load_class_names(CLASS_NAMES_PATH)
parent2children, child2parents = load_class_hierarchy(CLASS_HIERARCHY_PATH)
class_keywords = load_class_keywords(CLASS_KEYWORDS_PATH, class_names)

# Load pre-trained embeddings
label_data = torch.load(LABEL_EMB_PATH)
label_init_emb = label_data["embeddings"]

train_data = torch.load(TRAIN_EMB_PATH)
train_ids = train_data["ids"]
train_doc_embs = train_data["embeddings"]

test_data = torch.load(TEST_EMB_PATH)
test_ids = test_data["ids"]
test_doc_embs = test_data["embeddings"]

In [ ]:
# Train data + test data
# ("You are allowed to use the test corpus information during training.")
train_texts = [train_corpus[pid] for pid in train_ids]
test_texts = [test_corpus[pid] for pid in test_ids]
all_doc_texts = train_texts + test_texts

all_doc_embs = torch.cat([train_doc_embs, test_doc_embs], dim=0)
all_ids = list(range(len(all_doc_embs)))

## 2. Doc-label similarity score

In [ ]:
# Label texts
def build_label_texts(class_names, class_keywords):
    """
    Args: class_names, class_keywords
    Returns: label_texts (List with length C)
    ex) 'grocery gourmet food snacks condiments ...'
    """
    label_texts = []

    for cid, name in enumerate(class_names):
        pretty_name = name.replace("_", " ")
        keywords = class_keywords.get(cid, [])
        text = " ".join([pretty_name] + keywords)
        label_texts.append(text)

    return label_texts

label_texts = build_label_texts(class_names, class_keywords)

In [ ]:
# Label TF-IDF vectorizer
def build_label_tfidf(label_texts):
    """
    Args: label_texts (list with length C)
    Returns: vectorizer, label_tfidf (C x V sparse matrix)
    """
    vectorizer = TfidfVectorizer()
    label_tfidf = vectorizer.fit_transform(label_texts)
    return vectorizer, label_tfidf

# Semantic similarity (BERT embedding)
def compute_embedding_similarity_matrix(doc_embs, label_emb):
    """
    Args:
      - doc_embs: (N, D) torch.Tensor
      - label_emb: (C, D) torch.Tensor
    Returns: S_emb (N x C) numpy array (cosine similarity)
    """
    # L2 normalization
    doc_norm = torch.nn.functional.normalize(doc_embs, p=2, dim=1)
    label_norm = torch.nn.functional.normalize(label_emb, p=2, dim=1)
    S_emb = doc_norm @ label_norm.T

    return S_emb.cpu().numpy()

# Lexical similarity (TF-IDF)
def compute_lexical_similarity_matrix(doc_texts, vectorizer, label_tfidf, batch_size=2000):
    """
    Args:
      - doc_texts (list with length C)
      - vectorizer, label_tfidf
    Returns: S_lex (N x C) numpy array
    """
    sims_list = []
    for i in tqdm(range(0, len(doc_texts), batch_size), desc="Computing lexical similarity"):
        batch = doc_texts[i:i+batch_size]
        doc_vec = vectorizer.transform(batch)
        sims = cosine_similarity(doc_vec, label_tfidf)
        sims_list.append(sims)
    S_lex = np.vstack(sims_list)

    return S_lex

# Per-doc normalization
def normalize_per_doc(S, eps=1e-8):
    """
    Args: (N, C) numpy array
    Returns: normalized array
    """
    S_min = S.min(axis=1, keepdims=True)
    S_max = S.max(axis=1, keepdims=True)
    S_norm = (S - S_min) / (S_max - S_min + eps)
    return S_norm

In [ ]:
# Make weighted sum of scores S_total
def build_doc_label_scores(doc_embs, doc_texts, label_emb, label_texts, alpha=0.7):

    # Semantic similarity
    S_emb = compute_embedding_similarity_matrix(doc_embs, label_emb)
    S_emb_norm = normalize_per_doc(S_emb)

    # Lexical similarity
    vectorizer, label_tfidf = build_label_tfidf(label_texts)
    S_lex = compute_lexical_similarity_matrix(doc_texts, vectorizer, label_tfidf)
    S_lex_norm = normalize_per_doc(S_lex)

    # Weighted sum
    S_total = alpha * S_emb_norm + (1.0 - alpha) * S_lex_norm

    return S_total, S_emb_norm, S_lex_norm

In [ ]:
S_total, S_emb_norm, S_lex_norm = build_doc_label_scores(all_doc_embs, all_doc_texts,
                                                         label_init_emb, label_texts)

Computing lexical similarity: 100%|██████████| 25/25 [00:03<00:00,  7.43it/s]


In [ ]:
print(S_total.shape)

(49145, 531)


In [ ]:
print(S_total)

[[0.69435729 0.38670886 0.33116645 ... 0.36131629 0.17893836 0.2443772 ]
 [0.48584843 0.58682477 0.49649608 ... 0.30038199 0.41607911 0.26472083]
 [0.57875067 0.447613   0.49069276 ... 0.22441624 0.34733239 0.21145502]
 ...
 [0.34258714 0.35639098 0.31981689 ... 0.34828609 0.22608396 0.18286896]
 [0.28331432 0.31074825 0.24942918 ... 0.31920487 0.17704256 0.20545334]
 [0.30890104 0.24916907 0.21896467 ... 0.30150977 0.19628203 0.18232858]]


## 3. Check the confidence of silver labels

In [ ]:
# Confidence values from S_total
S_total_t = torch.from_numpy(S_total)
with torch.no_grad():
    conf_values, _ = S_total_t.max(dim=1)

# Threshold to trust silver labels
t_silver = 0.7

# Divide into Labeled/Unlabeled indices using confidence values
l_mask = conf_values >= t_silver
u_mask = ~l_mask

l_indices = l_mask.nonzero(as_tuple=True)[0]
u_indices = u_mask.nonzero(as_tuple=True)[0]

In [ ]:
print(f"#L (labeled)  : {len(l_indices)}")
print(f"#U (unlabeled): {len(u_indices)}")

#L (labeled)  : 40220
#U (unlabeled): 8925


In [ ]:
# Select samples among U to send to LLM
u_conf = conf_values[u_indices]
sorted_conf, sorted_idx = torch.sort(u_conf)

n_llm = 1000
u_llm_indices = u_indices[sorted_idx[:n_llm]]

**1000 LLM API calls but got the results of 250 calls due to restriction of usage**

## 4. LLM API calls

In [ ]:
# List of candidate label ids for each document
def get_topk_label_ids_for_doc(doc_idx, S_total, k=10):
    scores = S_total[doc_idx]
    topk_ids = np.argsort(-scores)[:k]
    return topk_ids.tolist()

# Generate the prompt
def build_prompt_for_doc(doc_idx, doc_text, candidate_ids):
    cand_lines = []
    for cid in candidate_ids:
        name = class_names[cid]
        desc = label_texts[cid]
        cand_lines.append(
            f"- id: {cid}, name: {name}, description: {desc}"
        )
    candidates_str = "\n".join(cand_lines)

    prompt = f"""
You are helping to classify Amazon products into multiple categories.

Document index: {doc_idx}

Product text:
\"\"\"{doc_text}\"\"\"

Candidate labels (each with id, name, and description):
{candidates_str}

Your task:
- Choose the best 2 or 3 labels from the candidate list that match this product.
- Only use the given label ids.
- If none of the candidates are appropriate, return an empty list.

Return the answer as a single JSON object, with this exact structure:
{{"labels": [id1, id2, id3]}}

Examples:
{{"labels": [17, 135]}}
{{"labels": []}}

Answer with JSON only, without any extra explanation.
"""
    return prompt.strip()

In [ ]:
# Prepare GEMINI client
client = genai.Client(api_key=GEMINI_API_KEY)

llm_model_name = "gemini-2.5-flash"

ROOT = Path("Amazon_products")
save_path = ROOT / "llm_labels_250.jsonl"

In [ ]:
# Loop of LLM API calls for u_llm_indices
import json

llm_results = []    # [{"doc_index": int, "labels": [int, ...]}, ...]
num_labels = len(class_names)
top_k = 10

for doc_idx_tensor in u_llm_indices:
    doc_idx = int(doc_idx_tensor.item())
    doc_text = all_doc_texts[doc_idx]
    candidate_ids = get_topk_label_ids_for_doc(doc_idx, S_total, k=top_k)
    user_prompt = build_prompt_for_doc(doc_idx, doc_text, candidate_ids)

    content = None

    try:
        response = client.models.generate_content(
            model=llm_model_name,
            contents=[user_prompt],
            config=GenerateContentConfig(response_mime_type="application/json")
        )

        if response.text is None:
            reason = response.prompt_feedback.block_reason.name if response.prompt_feedback else "UNKNOWN_ERROR"
            print(f"Doc {doc_idx} failed. (Text is None). {reason}")
            content = '{"labels": []}'
        else:
            content = response.text

    except Exception as e:
        print(f"Doc {doc_idx} error during API call: {e}")
        content = '{"labels": []}'

    parsed = None
    try:
        parsed = json.loads(content)

    except json.JSONDecodeError:
        start = content.find("{")
        end = content.rfind("}")

        if start != -1 and end != -1 and start <= end:
            try:
                parsed = json.loads(content[start : end + 1])
            except json.JSONDecodeError:
                print(f"Doc {doc_idx} 2nd JSON parsinf failed. Raw Content: {content[:100]}...")
                parsed = {"labels": []}
        else:
            print(f"Doc {doc_idx} cannot find JSON structure.")
            parsed = {"labels": []}

    labels = parsed.get("labels", [])
    labels = [int(x) for x in labels if isinstance(x, (int, str)) and 0 <= int(x) < num_labels]

    llm_results.append({
        "doc_index": doc_idx,
        "labels": labels,
    })

    with save_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(llm_results[-1], ensure_ascii=False) + "\n")

**1000 LLM API calls but got the results of 250 calls due to restriction of usage**

In [ ]:
llm_results

In [ ]:
# Load LLM result from jsonl
import json

ROOT = Path("Amazon_products")
save_path = ROOT / "llm_labels_250.jsonl"

llm_labels_dict = {}   # {doc_index: [label_ids]}

with save_path.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        doc_idx = int(obj["doc_index"])
        labels = [int(x) for x in obj.get("labels", [])]
        llm_labels_dict[doc_idx] = labels

u_llm_indices_final = []
for doc_idx_tensor in u_llm_indices:
    doc_idx = int(doc_idx_tensor.item())
    labels = llm_labels_dict.get(doc_idx, [])
    if len(labels) > 0:
        u_llm_indices_final.append(doc_idx)

print(f"#U_LLM with valid labels: {len(u_llm_indices_final)}")

#U_LLM with valid labels: 185
